# Ćwiczenie 4 — Matplotlib (wersja dydaktyczna) — wersja done

To jest **proponowana rewizja** ćwiczeń do wykładu 4.  
Cel nie jest tylko „narysować wykres”, ale nauczyć się:

- dobrać **typ wykresu do pytania analitycznego**,
- zbudować wykres **czytelny** (osie, tytuł, legenda, ticki, adnotacja),
- połączyć kilka wykresów w **dashboard**,
- rozumieć intuicję z wykładu 4, zwłaszcza:
  - `04.01` line plot,
  - `04.02` scatter,
  - `04.03` errorbars,
  - `04.05` histogramy oraz **krótkie mosty**: `ravel`, `reshape`, `meshgrid`,
  - `04.07` heatmapa / colorbar,
  - `04.08` multiple subplots,
  - `04.09–04.10` annotation + ticks,
  - `04.11` style.

## Jak czytać wersję done
- najpierw porównaj rozwiązanie z wersją studencką,
- po każdej funkcji zwróć uwagę, **jaki problem analityczny rozwiązuje dany wykres**,
- krótkie odpowiedzi tekstowe traktuj jako **element dydaktyczny**, nie formalność,
- sekcje oznaczone ⭐ są opcjonalne.

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from projekt_1_4_bikeshare_core import make_bikeshare_data

plt.rcParams["figure.figsize"] = (7, 4)
np.random.seed(0)

def assert_nonempty_str(x: str, *, min_len: int = 40):
    assert isinstance(x, str), f"Expected str, got {type(x)}"
    assert len(x.strip()) >= min_len, f"Text too short: {len(x.strip())} < {min_len}"

trips, stations, weather = make_bikeshare_data(n_trips=5000, n_stations=18, n_days=21, seed=0)

# Przygotowanie prostych tabel roboczych — celowo gotowe, bo tu ćwiczymy głównie Matplotlib.
trips = trips.copy()
trips["duration_min"] = (trips["end_ts"] - trips["start_ts"]).dt.total_seconds() / 60.0
trips["hour"] = trips["start_ts"].dt.hour
trips["weekday"] = pd.Categorical(
    trips["start_ts"].dt.day_name().str[:3],
    categories=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    ordered=True,
)
trips["date"] = trips["start_ts"].dt.floor("D")

daily = trips.groupby("date").size().rename("n_trips").to_frame()
daily["rolling_7d"] = daily["n_trips"].rolling(7, min_periods=1).mean()

station_stats = (
    trips.groupby("true_start_station_id")
    .agg(total_trips=("trip_id", "size"), mean_duration=("duration_min", "mean"))
    .merge(
        stations[["station_id", "capacity", "district"]],
        left_index=True,
        right_on="station_id",
        how="left",
    )
)

weekday_summary = (
    trips.groupby("weekday", observed=False)
    .agg(mean_duration=("duration_min", "mean"), std_duration=("duration_min", "std"), n=("trip_id", "size"))
    .reset_index()
)
weekday_summary["sem"] = weekday_summary["std_duration"] / np.sqrt(weekday_summary["n"].clip(lower=1))

heatmap_df = (
    trips.groupby(["weekday", "hour"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(24), fill_value=0)
)

print("trips:", trips.shape, "| stations:", stations.shape, "| weather:", weather.shape)
display(daily.head())
display(station_stats.head())
display(weekday_summary.head())
display(heatmap_df.iloc[:3, :6])

trips: (5000, 18) | stations: (18, 6) | weather: (504, 4)


,n_trips,rolling_7d
date,,
2024-05-01,238,238.000000
2024-05-02,250,244.000000
2024-05-03,230,239.333333
2024-05-04,264,245.500000
2024-05-05,223,241.000000


,total_trips,mean_duration,station_id,capacity,district
0,274,30.620983,0,21,Praga
1,302,23.982914,1,41,Mokotow
2,257,23.241098,2,19,Zachod
3,298,44.111438,3,17,Poludnie
4,267,21.799762,4,34,Poludnie


,weekday,mean_duration,std_duration,n,sem
0,Mon,27.298665,12.019677,692,0.456920
1,Tue,28.047626,12.726427,725,0.472648
2,Wed,28.604572,12.298090,749,0.449362
3,Thu,28.132777,12.796388,693,0.486095
4,Fri,27.959152,12.478934,703,0.470652


hour,0,1,2,3,4,5
weekday,,,,,,
Mon,8,2,3,2,3,10
Tue,5,6,2,6,5,5
Wed,4,2,2,3,4,10


## 0. Most z 04.05 — macierz, `ravel()`, `reshape()`, `meshgrid()`

W wykładzie 4 pojawiły się krótkie polskie dodatki o:
- `reshape`,
- `ravel`,
- `meshgrid`,
- `vstack`.

To **warto zachować**, bo pomaga zrozumieć:
- czym tak naprawdę jest **heatmapa** (macierz),
- jak przejść od siatki punktów do danych do rysowania,
- skąd biorą się kształty `(2, N)` i `(N, 2)`.

### Ćwiczenie 0.1 — Heatmapa jako macierz: `ravel()` → `reshape()`

Weź macierz `heatmap_df.to_numpy()` i:
1. spłaszcz ją do wektora 1D,
2. odtwórz z powrotem macierz o tym samym kształcie.

To jest prosty, ale ważny most między NumPy a wykresem typu `imshow`.

In [ ]:
def flatten_and_restore_heatmap(heatmap_df: pd.DataFrame):
    arr = heatmap_df.to_numpy()
    flat = arr.ravel()
    restored = flat.reshape(arr.shape)
    return flat, restored

flat, restored = flatten_and_restore_heatmap(heatmap_df)
arr = heatmap_df.to_numpy()

assert isinstance(flat, np.ndarray) and flat.ndim == 1
assert isinstance(restored, np.ndarray) and restored.shape == arr.shape
assert np.array_equal(restored, arr)
print("✅ Ćwiczenie 0.1")


### Ćwiczenie 0.2 — `meshgrid()` i dwie reprezentacje punktów ⭐

Dla siatki punktów chcemy zobaczyć dwie postacie:
- `(2, N)` — dobre np. do niektórych obliczeń numerycznych,
- `(N, 2)` — lista punktów, bardziej „intuicyjna”.

To ćwiczenie jest krótkim nawiązaniem do polskich dodatków w `04.05`.

In [ ]:
def meshgrid_points(xgrid: np.ndarray, ygrid: np.ndarray):
    X, Y = np.meshgrid(xgrid, ygrid)
    points_rows = np.vstack([X.ravel(), Y.ravel()])
    points_pairs = np.column_stack([X.ravel(), Y.ravel()])
    return X, Y, points_rows, points_pairs

xgrid = np.linspace(-2, 2, 6)
ygrid = np.linspace(-1, 1, 5)
X, Y, points_rows, points_pairs = meshgrid_points(xgrid, ygrid)

assert X.shape == (len(ygrid), len(xgrid))
assert Y.shape == (len(ygrid), len(xgrid))
assert points_rows.shape == (2, len(xgrid) * len(ygrid))
assert points_pairs.shape == (len(xgrid) * len(ygrid), 2)
print("✅ Ćwiczenie 0.2")


In [ ]:
WARMUP_NOTE = (
    "Macierz o kształcie (2, N) przechowuje współrzędne x i y jako dwa osobne wiersze, "
    "co bywa wygodne w niektórych obliczeniach numerycznych i przy pracy z siatką punktów. "
    "Postać (N, 2) oznacza listę N punktów w 2D i jest bardziej intuicyjna do interpretacji. "
    "W praktyce trzeba rozumieć obie reprezentacje, bo biblioteki numeryczne i wizualizacyjne "
    "nie zawsze oczekują danych w tym samym układzie."
)

assert_nonempty_str(WARMUP_NOTE, min_len=100)
WARMUP_NOTE


## 04.01 — Simple Line Plots

### Ćwiczenie 4.01.1 — Dzienna liczba przejazdów

Narysuj:
- linię `n_trips`,
- linię `rolling_7d`,
- tytuł, opisy osi i legendę,
- delikatną siatkę (`grid`).

**Cel dydaktyczny:** zobaczyć różnicę między surowym sygnałem a wygładzaniem.

In [ ]:
def plot_daily_trips(ax, daily: pd.DataFrame):
    ax.plot(daily.index, daily["n_trips"], marker="o", linewidth=1.5, label="n_trips")
    ax.plot(daily.index, daily["rolling_7d"], linewidth=2.5, label="rolling_7d")
    ax.set_title("Dzienna liczba przejazdów")
    ax.set_xlabel("data")
    ax.set_ylabel("liczba przejazdów")
    ax.legend()
    ax.grid(True, alpha=0.3)



fig, ax = plt.subplots()
plot_daily_trips(ax, daily)
labels = [line.get_label() for line in ax.lines]
assert len(ax.lines) >= 2
assert "n_trips" in labels and "rolling_7d" in labels
assert ax.get_title() != ""
assert ax.get_xlabel() != "" and ax.get_ylabel() != ""
plt.show()
print("✅ Ćwiczenie 4.01.1")


In [ ]:
NOTE_LINE = (
    "Wykres liniowy dobrze pokazuje zmianę popytu w czasie, a średnia krocząca 7D tłumi "
    "przypadkowe wahania i ujawnia trend. To ważne, bo w projekcie chcemy odróżniać sygnał "
    "od szumu. Gdyby pokazać tylko wartości dzienne, łatwiej byłoby przegapić ogólny kierunek zmian."
)

assert_nonempty_str(NOTE_LINE, min_len=120)
NOTE_LINE


## 04.02 — Simple Scatter Plots

### Ćwiczenie 4.02.1 — Popyt vs pojemność stacji

Wykres scatter:
- `x = capacity`,
- `y = total_trips`,
- kolor punktu zależy od `mean_duration`,
- dodaj colorbar,
- dodaj opisy osi i tytuł.

**Cel dydaktyczny:** jeden wykres ma pokazać relację między dwiema zmiennymi,
a trzecią zakodować kolorem.

In [ ]:
def plot_station_scatter(ax, station_stats: pd.DataFrame):
    sc = ax.scatter(
        station_stats["capacity"],
        station_stats["total_trips"],
        c=station_stats["mean_duration"],
        s=70,
        alpha=0.8,
    )
    ax.set_title("Popyt vs pojemność stacji")
    ax.set_xlabel("capacity")
    ax.set_ylabel("total_trips")
    ax.grid(True, alpha=0.25)
    ax.figure.colorbar(sc, ax=ax, label="mean_duration [min]")



fig, ax = plt.subplots()
plot_station_scatter(ax, station_stats)
assert len(ax.collections) >= 1
assert ax.get_xlabel() == "capacity"
assert ax.get_ylabel() == "total_trips"
assert len(fig.axes) >= 2  # drugi axes to colorbar
plt.show()
print("✅ Ćwiczenie 4.02.1")


In [ ]:
NOTE_SCATTER = (
    "Scatter jest dobrym wyborem, bo porównujemy dwie zmienne ilościowe dla wielu stacji i "
    "szukamy zależności, skupień albo odstających punktów. Kolor pozwala dołożyć trzeci wymiar "
    "informacji bez rozbijania analizy na kilka osobnych wykresów. To lepsze niż np. słupki, "
    "bo nie narzuca sztucznego porządku obserwacji."
)

assert_nonempty_str(NOTE_SCATTER, min_len=120)
NOTE_SCATTER


### Ćwiczenie 4.02A / 04.08A — Kowariancja a kształt chmury punktów ⭐

W polskim dodatku w `04.08` pojawił się bardzo dobry motyw:  
**znak kowariancji zmienia orientację chmury punktów**.

To jest świetne uzupełnienie sekcji o scatterze, dlatego warto je zostawić jako krótki add-on.

In [ ]:
def plot_covariance_demo(seed: int = 0):
    rng = np.random.default_rng(seed)
    covs = [
        np.array([[1, 0], [0, 2]], dtype=float),
        np.array([[1, 1], [1, 2]], dtype=float),
        np.array([[1, -1], [-1, 2]], dtype=float),
    ]
    titles = ["cov = 0", "cov > 0", "cov < 0"]

    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, cov, title in zip(axes, covs, titles):
        pts = rng.multivariate_normal(mean=[0, 0], cov=cov, size=500)
        ax.scatter(pts[:, 0], pts[:, 1], s=12, alpha=0.45)
        ax.set_title(title)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.grid(True, alpha=0.2)
    return fig, axes

fig, axes = plot_covariance_demo(seed=0)
assert len(axes) == 3
for ax in axes:
    assert len(ax.collections) >= 1
plt.show()
print("✅ Ćwiczenie 4.02A")


## 04.03 — Errorbars

### Ćwiczenie 4.03.1 — Średni czas przejazdu ± SEM

Użyj `weekday_summary` i narysuj średni czas przejazdu z errorbarami `sem`.

**Cel dydaktyczny:** student ma zobaczyć, że średnia bez niepewności bywa myląca.

In [ ]:
def plot_duration_errorbars(ax, weekday_summary: pd.DataFrame):
    ax.errorbar(
        weekday_summary["weekday"],
        weekday_summary["mean_duration"],
        yerr=weekday_summary["sem"],
        fmt="-o",
        capsize=4,
    )
    ax.set_title("Średni czas przejazdu ± SEM")
    ax.set_xlabel("weekday")
    ax.set_ylabel("duration [min]")
    ax.grid(True, alpha=0.3)



fig, ax = plt.subplots()
plot_duration_errorbars(ax, weekday_summary)
assert len(ax.lines) >= 1
assert ax.get_title() != ""
assert ax.get_xlabel() != "" and ax.get_ylabel() != ""
plt.show()
print("✅ Ćwiczenie 4.03.1")


## 04.05 — Histograms and Binnings

### Ćwiczenie 4.05.1 — Histogram czasów przejazdu

Narysuj histogram `duration_min`:
- z sensownym zakresem (np. do 60 min),
- z opisami osi,
- z tytułem.

**Cel dydaktyczny:** rozpoznać rozkład i ewentualną prawoskośność.

In [ ]:
def plot_duration_hist(ax, trips: pd.DataFrame):
    data = trips.loc[trips["duration_min"] <= 60, "duration_min"]
    ax.hist(data, bins=30, alpha=0.85)
    ax.set_title("Histogram czasów przejazdu")
    ax.set_xlabel("duration [min]")
    ax.set_ylabel("count")



fig, ax = plt.subplots()
plot_duration_hist(ax, trips)
assert len(ax.patches) > 0
assert ax.get_title() != ""
assert ax.get_xlabel() != "" and ax.get_ylabel() != ""
plt.show()
print("✅ Ćwiczenie 4.05.1")


In [ ]:
NOTE_HIST = (
    "Histogram pokazuje kształt rozkładu, a nie tylko pojedynczą liczbę opisową. W danych czasów "
    "przejazdu zwykle widać prawoskośność: większość przejazdów jest dość krótka, ale pojawia się "
    "ogon dłuższych obserwacji. To ważne, bo sama średnia może wtedy słabo oddawać typowy przypadek."
)

assert_nonempty_str(NOTE_HIST, min_len=120)
NOTE_HIST


## 04.07 — Customizing Colorbars

### Ćwiczenie 4.07.1 — Heatmapa: weekday × hour

Pokaż liczbę przejazdów jako heatmapę:
- wiersze: dni tygodnia,
- kolumny: godziny,
- dodaj colorbar,
- podpisz ticki na obu osiach.

**Cel dydaktyczny:** heatmapa jest tu lepsza niż 168 osobnych słupków,
bo pokazuje wzorzec tygodniowo-godzinowy jako jedną strukturę.

In [ ]:
def plot_weekday_hour_heatmap(ax, heatmap_df: pd.DataFrame):
    im = ax.imshow(heatmap_df.to_numpy(), aspect="auto", origin="upper")
    ax.set_title("Liczba przejazdów: weekday × hour")
    ax.set_xlabel("hour")
    ax.set_ylabel("weekday")
    ax.set_xticks(np.arange(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns)
    ax.set_yticks(np.arange(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index)
    ax.figure.colorbar(im, ax=ax, label="n_trips")



fig, ax = plt.subplots(figsize=(9, 4))
plot_weekday_hour_heatmap(ax, heatmap_df)
assert (len(ax.images) + len(ax.collections)) >= 1
assert ax.get_title() != ""
assert len(ax.get_xticks()) > 0 and len(ax.get_yticks()) > 0
assert len(fig.axes) >= 2  # heatmap + colorbar
plt.show()
print("✅ Ćwiczenie 4.07.1")


In [ ]:
NOTE_HEATMAP = (
    "Heatmapa pozwala natychmiast zobaczyć, czy popyt zależy od dnia tygodnia i godziny. "
    "W jednym obrazie porównujemy cały tydzień i od razu szukamy pasów wysokiej aktywności, "
    "np. porannego lub popołudniowego szczytu. Taki wzorzec byłby dużo mniej czytelny na szeregu "
    "pojedynczych wykresów słupkowych."
)

assert_nonempty_str(NOTE_HEATMAP, min_len=120)
NOTE_HEATMAP


## 04.08 — Multiple Subplots

### Ćwiczenie 4.08.1 — Dashboard 2×2

Zbuduj figure 2×2 zawierającą:
1. line plot,
2. scatter,
3. errorbars,
4. heatmapę albo histogram.

Funkcja ma zwracać `(fig, axes)`.

**Cel dydaktyczny:** student ma zobaczyć, że pojedyncze wykresy odpowiadają na różne pytania,
a dashboard scala je w jedną narrację analityczną.

In [ ]:
def build_dashboard(daily, station_stats, weekday_summary, heatmap_df, trips):
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    plot_daily_trips(axes[0, 0], daily)
    plot_station_scatter(axes[0, 1], station_stats)
    plot_duration_errorbars(axes[1, 0], weekday_summary)
    plot_weekday_hour_heatmap(axes[1, 1], heatmap_df)
    return fig, axes



fig, axes = build_dashboard(daily, station_stats, weekday_summary, heatmap_df, trips)
assert hasattr(fig, "axes")
assert axes.shape == (2, 2)
assert len(fig.axes) >= 4
plt.show()
print("✅ Ćwiczenie 4.08.1")


## 04.09–04.10 — Annotation + Ticks

### Ćwiczenie 4.09.1 — Zaznacz najbardziej ruchliwy dzień

Na line plocie dziennych przejazdów dodaj:
- adnotację dla dnia o maksymalnym `n_trips`,
- czytelne etykiety osi X (obracanie ticków, jeśli trzeba).

Dobrze, jeśli tekst adnotacji pokazuje **datę i wartość**.

In [ ]:
def annotate_busiest_day(ax, daily: pd.DataFrame):
    idx = daily["n_trips"].idxmax()
    val = daily.loc[idx, "n_trips"]
    label = f"{idx.date()}: {int(val)}"
    ax.annotate(
        label,
        xy=(idx, val),
        xytext=(10, 15),
        textcoords="offset points",
        arrowprops=dict(arrowstyle="->"),
    )
    for tick in ax.get_xticklabels():
        tick.set_rotation(30)
        tick.set_ha("right")



fig, ax = plt.subplots()
plot_daily_trips(ax, daily)
annotate_busiest_day(ax, daily)
best_val = int(daily["n_trips"].max())
assert len(ax.texts) >= 1
assert any(str(best_val) in t.get_text() for t in ax.texts)
plt.show()
print("✅ Ćwiczenie 4.09.1")


## 04.11 — Stylesheets ⭐

### Ćwiczenie 4.11.1 — Jednolity styl

Wybierz jeden prosty styl (`default`, `classic`, `ggplot`) i zapisz
2–4 zdania, jak zmienił czytelność dashboardu.

To ćwiczenie zostawiłbym jako **opcjonalne**:
ważniejsze od dekoracji są poprawne osie, etykiety i dobór wykresu.

In [ ]:
STYLE_NOTE = (
    "Porównałem style i w tym typie dashboardu najbardziej odpowiada mi styl `default`, "
    "bo jest neutralny i nie odciąga uwagi od danych. `ggplot` może być przyjemny wizualnie, "
    "ale czasem wprowadza zbyt silną warstwę dekoracyjną. W zadaniach analitycznych ważniejsze "
    "jest, żeby odbiorca szybko odczytał wzorzec niż żeby wykres wyglądał efektownie."
)

assert_nonempty_str(STYLE_NOTE, min_len=90)
STYLE_NOTE


## 04.X — Dobór wykresu do pytania

To jest brakujący element w obecnym zestawie: student powinien umieć nie tylko
**wykonać** wykres, ale też **wybrać** właściwy typ wykresu.

Dla każdego pytania wpisz najlepszy typ wykresu:
- `line`,
- `scatter`,
- `histogram`,
- `heatmap`.

Możesz użyć dokładnie tych nazw.

In [ ]:
CHART_CHOICES = {
    "trend_over_time": "line",
    "capacity_vs_demand": "scatter",
    "duration_distribution": "histogram",
    "weekday_hour_pattern": "heatmap",
}

CHART_CHOICES_NOTE = (
    "Dobór wykresu powinien wynikać z pytania analitycznego, a nie z przyzwyczajenia albo estetyki. "
    "Jeśli interesuje nas zmiana w czasie, najlepszy jest line plot, bo pokazuje ciągłość i trend. "
    "Jeśli porównujemy dwie zmienne ilościowe, naturalny jest scatter. Rozkład jednej zmiennej najczytelniej "
    "pokazuje histogram, a wzorzec na siatce dwóch wymiarów kategorialno-czasowych dobrze ujmuje heatmapa. "
    "W praktyce zły typ wykresu może ukryć strukturę danych nawet wtedy, gdy sam wykres jest wykonany poprawnie technicznie."
)

allowed = {"line", "scatter", "histogram", "heatmap"}
assert set(CHART_CHOICES.keys()) == {
    "trend_over_time",
    "capacity_vs_demand",
    "duration_distribution",
    "weekday_hour_pattern",
}
assert set(CHART_CHOICES.values()).issubset(allowed)
assert CHART_CHOICES["trend_over_time"] == "line"
assert CHART_CHOICES["capacity_vs_demand"] == "scatter"
assert CHART_CHOICES["duration_distribution"] == "histogram"
assert CHART_CHOICES["weekday_hour_pattern"] == "heatmap"
assert_nonempty_str(CHART_CHOICES_NOTE, min_len=180)
CHART_CHOICES, CHART_CHOICES_NOTE
